# 基于宝可梦种族值的逻辑回归进化阶段预测模型

**作者**: KcirtW0009  
**邮箱**: KcirtW0009@outlook.com  
**项目**: LogisticRegressionPrediction-EvolutionaryStage-by-PokemonStats

---

## 项目概述

本项目使用**逻辑回归（Logistic Regression）**机器学习算法，通过分析宝可梦的六项基础数值（HP、攻击、防御、特攻、特防、速度），自动预测该宝可梦所处的进化阶段。

### 核心功能

- 📊 **数据预处理与清洗**：解析原始CSV数据，过滤有效形态
- 🔗 **进化图构建**：基于BFS广度优先搜索计算全局进化深度
- 🎯 **多分类预测**：支持四类进化阶段分类任务
- 📈 **模型评估与可视化**：提供全面的性能评估指标
- 🎮 **交互式预测界面**：支持多种输入方式进行实时预测

---

In [ ]:
# ============================================================================
# 第一部分：环境配置与依赖导入
# ============================================================================

import pandas as pd                    # 数据处理与分析库
import numpy as np                     # 数值计算库
import matplotlib.pyplot as plt        # 数据可视化库
import seaborn as sns                  # 统计图形增强库
from sklearn.model_selection import train_test_split  # 数据集划分工具
from sklearn.preprocessing import StandardScaler       # 数据标准化处理器
from sklearn.linear_model import LogisticRegression    # 逻辑回归分类器
from sklearn.pipeline import Pipeline                  # 机器学习流水线
from sklearn.metrics import (                         # 模型评估指标
    classification_report,      # 分类报告（精确率、召回率、F1分数）
    confusion_matrix,           # 混淆矩阵
    accuracy_score              # 准确率
)

# 配置matplotlib支持中文显示
plt.rcParams['font.sans-serif'] = ['SimHei', 'Microsoft YaHei', 'DejaVu Sans']
plt.rcParams['axes.unicode_minus'] = False  # 解决负号显示问题

print("✅ 环境配置完成，所有依赖库导入成功")

In [ ]:
# ============================================================================
# 第二部分：数据加载与初步过滤
# ============================================================================

# 加载宝可梦数据集
# 数据来源：Pokemon Showdown Pokedex
# 包含字段：num(编号), name(名称), forme(形态), baseStats(基础数值), types(属性), evos(进化后), prevo(进化前)
df = pd.read_csv('Pokemon_Showdown_pokedex.csv')

# 步骤1：只保留正式宝可梦（全国图鉴编号 > 0）
# 过滤掉特殊形态、幻之宝可梦等非正式图鉴条目
df = df[df['num'] > 0].copy()

# 步骤2：处理缺失的形态信息
# 将NaN替换为空字符串，便于后续统一处理
df['forme'] = df['forme'].fillna('')

# 步骤3：定义允许的地区形态列表
# 只保留主要游戏地区的官方形态（阿罗拉、伽勒尔、洗翠、帕底亚）
allowed_formes = ['', 'Alola', 'Galar', 'Hisui', 'Paldea']

# 步骤4：执行形态过滤
# 移除Mega进化、超极巨化等非标准形态
df_filtered = df[df['forme'].isin(allowed_formes)]

# 更新主数据框并输出统计信息
df = df_filtered
print(f"📊 数据过滤统计:")
print(f"   - 过滤前记录数: {len(pd.read_csv('Pokemon_Showdown_pokedex.csv')[pd.read_csv('Pokemon_Showdown_pokedex.csv')['num'] > 0])}")
print(f"   - 过滤后记录数: {len(df)}")
print(f"\n📋 形态分布（前20个）:")
print(df['forme'].value_counts(dropna=False).head(20))

In [ ]:
# ============================================================================
# 第三部分：特征工程 - 解析种族值与属性
# ============================================================================

import ast  # 用于安全解析字符串形式的字典/列表

def parse_stats(stat_str):
    """
    解析宝可梦基础数值字符串
    
    参数:
        stat_str (str): 包含六项数值的字典字符串，如 "{hp:45, atk:49, ...}"
    
    返回:
        dict: 包含六个键值对的字典 {'hp': float, 'atk': float, ...}
    
    说明:
        - 使用ast.literal_eval安全解析，避免eval的安全风险
        - 缺失的数值填入NaN，保持数据完整性
    """
    try:
        stats = ast.literal_eval(stat_str)
        return {
            'hp': float(stats.get('hp', float('nan'))),      # 生命值
            'atk': float(stats.get('atk', float('nan'))),     # 物理攻击
            'def': float(stats.get('def', float('nan'))),     # 物理防御
            'spa': float(stats.get('spa', float('nan'))),     # 特殊攻击
            'spd': float(stats.get('spd', float('nan'))),     # 特殊防御
            'spe': float(stats.get('spe', float('nan')))      # 速度
        }
    except:
        # 解析失败时返回全NaN字典
        return {k: float('nan') for k in ['hp', 'atk', 'def', 'spa', 'spd', 'spe']}

# 逐行解析baseStats列，确保索引完全对齐
# 使用原DataFrame的索引遍历，避免数据错位问题
stats_list = []
for idx in df.index:
    row = df.loc[idx]
    stats = parse_stats(row['baseStats'])
    stats_list.append(stats)

# 构建统计数据DataFrame并与原数据合并
stats_df = pd.DataFrame(stats_list, index=df.index)  # 关键：使用相同索引保证对齐
stats_df = stats_df.astype(float)                     # 强制转换为浮点数类型

# 将解析后的数值列直接赋值到主DataFrame
df['hp'] = stats_df['hp']
df['atk'] = stats_df['atk']
df['def'] = stats_df['def']
df['spa'] = stats_df['spa']
df['spd'] = stats_df['spd']
df['spe'] = stats_df['spe']

# 计算总种族值（Base Stat Total）
# BST是衡量宝可梦整体实力的重要指标
df['bst'] = df['hp'] + df['atk'] + df['def'] + df['spa'] + df['spd'] + df['spe']

def parse_types(types_str):
    """
    解析属性字符串
    
    参数:
        types_str (str): 属性列表字符串，如 "['Fire', 'Flying']"
    
    返回:
        list: 属性名称列表
    """
    try:
        return ast.literal_eval(types_str)
    except:
        return []

# 解析属性信息并拆分为第一属性和第二属性
df['types'] = df['types'].apply(parse_types)
df['type1'] = df['types'].apply(lambda x: x[0] if len(x) > 0 else None)  # 主属性
df['type2'] = df['types'].apply(lambda x: x[1] if len(x) > 1 else None)  # 副属性

print("✅ 特征解析完成，示例数据前6行：")
display(df[['name', 'forme', 'hp', 'atk', 'spe', 'bst']].head(6))

In [ ]:
# ============================================================================
# 第四部分：进化图构建与阶段标注 - 核心算法
# ============================================================================

from collections import deque  # 双端队列，用于BFS算法

# 预处理：确保进化关系字段无缺失值
df['evos'] = df['evos'].fillna('')
df['prevo'] = df['prevo'].fillna('')

# ---------- 1. 构建全局进化有向图（父节点 -> 子节点）----------
# 图结构表示：key=宝可梦名称, value=进化后的子代列表
graph = {name: [] for name in df['name']}
for _, row in df.iterrows():
    parent = row['name']
    evos_str = row['evos']
    if evos_str not in ('', '[]'):
        try:
            children = ast.literal_eval(evos_str)
            if isinstance(children, list):
                graph[parent] = children
        except:
            pass  # 解析失败则跳过

# ---------- 2. 计算每个节点的入度（被进化的次数）----------
# 入度为0的节点可能是：基础形态或不进化的独立宝可梦
all_names = set(df['name'])
in_degree = {name: 0 for name in all_names}
for parent, children in graph.items():
    for child in children:
        if child in in_degree:
            in_degree[child] += 1

# ---------- 3. BFS广度优先搜索计算进化深度----------
# 时间复杂度: O(V+E)，V=顶点数，E=边数
# 空间复杂度: O(V)，用于存储队列和深度字典
depth = {}
queue = deque()

# 初始化：找到所有入度为0的根节点
for name in all_names:
    if in_degree[name] == 0:
        # 区分两种情况：
        # - 有后代 -> 是某条进化链的基础形态（depth=0）
        # - 无后代 -> 不进化的独立宝可梦（depth=-1）
        if len(graph[name]) > 0:
            depth[name] = 0          # 基础形态
            queue.append(name)
        else:
            depth[name] = -1         # 不进化

# BFS遍历计算每个节点的进化深度
while queue:
    cur = queue.popleft()
    for child in graph.get(cur, []):
        if child not in depth:
            depth[child] = depth[cur] + 1  # 子节点深度 = 父节点深度 + 1
            queue.append(child)

# 兜底处理：未访问到的节点标记为不进化
# 可能原因：孤立节点、数据异常或循环引用
for name in all_names:
    if name not in depth:
        depth[name] = -1

# ---------- 4. 将数值深度映射为人类可读的阶段标签----------
def depth_to_stage(d):
    """
    将进化深度转换为中文阶段标签
    
    映射规则:
        -1 → 不进化（如皮卡丘、传说宝可梦）
         0 → 基础形态（进化链起点，如小火龙）
         1 → 一阶进化（第一次进化后，如火恐龙）
        >=2 → 二阶及以上进化（最终形态，如喷火龙）
    
    参数:
        d (int): 进化深度值
    
    返回:
        str: 中文阶段标签
    """
    if d == -1:
        return '不进化'
    elif d == 0:
        return '基础'
    elif d == 1:
        return '一阶'
    else:
        return '二阶'   # d >= 2 的所有情况归为一类

# 为每只宝可梦标注进化阶段
df['stage'] = df['name'].map(lambda n: depth_to_stage(depth.get(n, -1)))

print("✅ 进化阶段标注完成（基于全局BFS算法）：")
print("\n📊 各阶段样本分布：")
print(df['stage'].value_counts())

In [ ]:
# ============================================================================
# 第五部分：进化阶段验证测试
# ============================================================================

# 选择代表性宝可梦进行验证
# 覆盖场景：完整三段进化、地区形态差异、分支进化、不进化个体
tests = [
    'Charmander', 'Charmeleon', 'Charizard',      # 完整三阶进化链（喷火龙系）
    'Arcanine', 'Arcanine-Hisui',                   # 地区形态对比（普通 vs 洗翠风速狗）
    'Qwilfish', 'Qwilfish-Hisui', 'Overqwil',       # 特殊进化案例（千针鱼→千面避役）
    'Pikachu', 'Raichu',                            # 两阶进化（皮卡丘系）
    'Mewtwo'                                        # 传说宝可梦（不进化）
]

print("🔍 进化阶段验证结果：")
verification_df = df[df['name'].isin(tests)][['name', 'forme', 'stage']]
display(verification_df)

# 验证要点检查
print("\n✅ 验证要点：")
print("   - Charmander(基础) → Charmeleon(一阶) → Charizard(二阶) ✓")
print("   - Arcanine(不进化) 与 Arcanine-Hisui(不进化) 应一致 ✓")
print("   - Qwilfish(不进化) → Overqwil(一阶) 特殊情况 ✓")
print("   - Mewtwo 应标记为'不进化' ✓")

In [ ]:
# ============================================================================
# 第六部分：高级特征工程
# ============================================================================

# ---------- 构造组合特征 ----------
# 设计思路：简化特征空间同时保留关键信息
# 物攻/特攻 和 物防/特防 往往呈现负相关（偏科现象）
# 取两者较大值可以更好地反映宝可梦的进攻/防守倾向

df['best_atk'] = df[['atk', 'spa']].max(axis=1)      # 最高攻击力（物攻vs特攻取大）
df['best_def'] = df[['def', 'spd']].max(axis=1)      # 最高防御力（物防vs特防取大）

# ---------- 选定最终特征集 ----------
# 特征选择理由：
# 1. HP - 生存能力的基础指标
# 2. best_atk - 综合进攻能力（避免物攻/特攻的多重共线性）
# 3. best_def - 综合防守能力（同上）
# 4. spe - 速度决定先手权，对战中的关键属性
# 
# 排除BST的原因：与其他6项高度相关，会造成冗余
feature_cols = ['hp', 'best_atk', 'best_def', 'spe']
X = df[feature_cols]  # 特征矩阵
y = df['stage']        # 目标变量（进化阶段）

print("📐 最终特征集设计：")
print(f"   特征数量: {len(feature_cols)} 个")
print(f"   样本数量: {len(X)} 个")
print("\n特征矩阵预览（前5行）：")
display(X.head())
print("\n目标变量分布：")
print(y.value_counts())

In [ ]:
# ============================================================================
# 第七部分：模型训练与Pipeline构建
# ============================================================================

# ---------- 数据集划分 ----------
# 使用分层抽样(stratify)保证各阶段比例在训练集和测试集中一致
# 测试集占比25%，训练集75%
# 随机种子42确保结果可复现
X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.25,        # 测试集比例
    random_state=42,       # 随机种子
    stratify=y             # 分层抽样，保持类别分布
)

# ---------- 构建机器学习Pipeline ----------
# Pipeline优势：
# 1. 避免数据泄漏（标准化只在训练集fit）
# 2. 代码简洁，便于部署
# 3. 方便超参数调优
#
# Pipeline流程：
# 原始数据 → StandardScaler(标准化) → LogisticRegression(分类预测)
model = Pipeline([
    (
        'scaler',
        StandardScaler()  # Z-score标准化：(x - mean) / std
    ),
    (
        'clf',
        LogisticRegression(
            multi_class='multinomial',  # 多项式逻辑回归（softmax）
            solver='lbfgs',            # L-BFGS优化器（适合中小规模数据）
            max_iter=2000,              # 最大迭代次数（确保收敛）
            class_weight='balanced',    # 自动处理类别不平衡
            random_state=42             # 随机种子
        )
    )
])

# 训练模型
model.fit(X_train, y_train)

print("✅ 模型训练完成！")
print(f"\n📊 训练集规模: {X_train.shape[0]} 样本")
print(f"📊 测试集规模: {X_test.shape[0]} 样本")
print(f"📊 特征维度: {X_train.shape[1]} 维")

In [ ]:
# ============================================================================
# 第八部分：模型评估与可视化
# ============================================================================

# ---------- 生成预测结果 ----------
y_train_pred = model.predict(X_train)  # 训练集预测
y_test_pred = model.predict(X_test)    # 测试集预测

# ---------- 输出准确率 ----------
train_acc = accuracy_score(y_train, y_train_pred)
test_acc = accuracy_score(y_test, y_test_pred)

print(f"🎯 模型性能评估：")
print(f"   训练集准确率: {train_acc:.3f} ({train_acc*100:.1f}%)")
print(f"   测试集准确率: {test_acc:.3f} ({test_acc*100:.1f}%)")
print()

# ---------- 详细分类报告 ----------
# 包含精确率(Precision)、召回率(Recall)、F1分数(F1-score)
# 以及各类别的支持数(Support)
print("📋 分类报告（测试集）：")
print(classification_report(y_test, y_test_pred))

# ---------- 混淆矩阵可视化 ----------
# 混淆矩阵展示各类别之间的混淆情况
# 对角线元素越大，说明分类越准确
cm = confusion_matrix(y_test, y_test_pred, labels=model.classes_)

plt.figure(figsize=(8, 6))
sns.heatmap(
    cm,
    annot=True,                # 显示数值
    fmt='d',                   # 整数格式
    cmap='Blues',              # 蓝色配色方案
    xticklabels=model.classes_, # X轴标签（预测值）
    yticklabels=model.classes_  # Y轴标签（真实值）
)
plt.xlabel('预测标签 (Predicted)', fontsize=12)
plt.ylabel('真实标签 (Actual)', fontsize=12)
plt.title('进化阶段预测混淆矩阵\nConfusion Matrix for Evolution Stage Prediction', fontsize=14)
plt.tight_layout()
plt.show()

In [ ]:
# ============================================================================
# 第九部分：特征权重分析与解释
# ============================================================================

# ---------- 提取模型系数 ----------
# 逻辑回归的系数反映了各特征对预测结果的贡献程度
# 正系数：特征值增大，属于该类的概率增加
# 负系数：特征值增大，属于该类的概率减小
if hasattr(model, 'named_steps'):
    clf = model.named_steps['clf']  # 从Pipeline中提取分类器
else:
    clf = model

coef = clf.coef_            # 形状: (n_classes, n_features)
intercept = clf.intercept_  # 形状: (n_classes,)
classes = clf.classes_      # 类别标签数组

# ---------- 创建系数DataFrame便于分析 ----------
coef_df = pd.DataFrame(coef, columns=feature_cols, index=classes)
print("📊 各进化阶段的特征系数表：")
print("（正值表示该特征促进归属此类别，负值表示抑制）")
display(coef_df)

# ---------- 系数可视化 ----------
plt.figure(figsize=(12, 7))
coef_df.T.plot(kind='bar', width=0.8)
plt.title('特征权重对比：不同进化阶段的影响因素\nFeature Weights Comparison Across Evolution Stages', fontsize=14)
plt.xlabel('特征名称 (Feature)', fontsize=12)
plt.ylabel('系数值 (Coefficient Value)', fontsize=12)
plt.axhline(y=0, color='black', linewidth=0.8, linestyle='--')  # 零线参考
plt.xticks(rotation=0)
plt.legend(title='进化阶段 (Stage)', bbox_to_anchor=(1.02, 1), loc='upper left')
plt.tight_layout()
plt.show()

# ---------- 业务解读 ----------
print("\n💡 关键发现与业务解读：")
print("   1. 通常'基础'形态的特征系数为负（各项数值较低）")
print("   2. '二阶'形态的best_atk/best_def系数通常为正（进化后能力提升）")
print("   3. 速度(spe)对高阶进化的区分度通常较高")
print("   4. 不进化群体可能包含两类：弱小宝可梦(低数值)和传说宝可梦(极高数值)")

In [ ]:
# ============================================================================
# 第十部分：交互式预测系统
# ============================================================================

from IPython.display import display, clear_output
import ipywidgets as widgets

# 验证模型支持概率输出
if hasattr(model, 'predict_proba'):
    predictor = model
else:
    raise AttributeError("当前模型不支持概率预测，请确认使用了正确的分类器")

def predict_and_display(hp, best_atk, best_def, spe):
    """
    执行预测并格式化显示结果
    
    参数:
        hp (float): 生命值
        best_atk (float): 最高攻击力
        best_def (float): 最高防御力
        spe (float): 速度
    
    返回:
        tuple: (预测标签, 概率字典)
    """
    # 构造输入DataFrame，必须保持列名与训练时一致
    input_df = pd.DataFrame([[hp, best_atk, best_def, spe]], columns=feature_cols)
    
    # 获取预测结果和概率分布
    pred = predictor.predict(input_df)[0]
    proba = predictor.predict_proba(input_df)[0]
    proba_dict = {cls: prob for cls, prob in zip(predictor.classes_, proba)}
    
    # 格式化输出
    print(f"🔹 预测进化阶段：{pred}")
    print("\n各阶段概率分布：")
    for cls, prob in sorted(proba_dict.items(), key=lambda x: x[1], reverse=True):
        bar = '█' * int(prob * 50)  # 简单的进度条可视化
        print(f"  {cls:6s}: {prob*100:5.1f}% {bar}")
    
    return pred, proba_dict

# ========== 方式1：手动输入特征值 ==========
print("=" * 60)
print("📊 预测方式1：手动输入特征值")
print("=" * 60)

# 创建输入控件
hp_input = widgets.FloatText(value=60, description='HP:', step=1)
atk_input = widgets.FloatText(value=80, description='最高攻击:', step=1)
def_input = widgets.FloatText(value=90, description='最高防御:', step=1)
spe_input = widgets.FloatText(value=70, description='速度:', step=1)
btn_manual = widgets.Button(description='🔮 开始预测', button_style='primary')
out_manual = widgets.Output()

def on_click_manual(b):
    """手动预测按钮回调函数"""
    with out_manual:
        clear_output(wait=True)
        print("\n⏳ 正在预测...")
        predict_and_display(
            hp_input.value,
            atk_input.value,
            def_input.value,
            spe_input.value
        )

btn_manual.on_click(on_click_manual)
display(widgets.VBox([hp_input, atk_input, def_input, spe_input, btn_manual, out_manual]))

In [ ]:
# ========== 方式2：按宝可梦名称查询 ==========
print("\n" + "=" * 60)
print("🔍 预测方式2：按名称查询（支持模糊匹配）")
print("=" * 60)

# 创建查询控件
name_input = widgets.Text(
    placeholder='输入英文名，如 Charizard, Pikachu...',
    description='名称:'
)
btn_name = widgets.Button(description='🔎 查询并预测', button_style='success')
out_name = widgets.Output()

def on_click_name(b):
    """名称查询按钮回调函数"""
    with out_name:
        clear_output(wait=True)
        name = name_input.value.strip().lower()
        
        # 输入验证
        if not name:
            print("⚠️ 请输入宝可梦名称！")
            return
        
        # 模糊匹配搜索（大小写不敏感）
        matches = df[df['name'].str.lower().str.contains(name, na=False)]
        
        if matches.empty:
            print(f"❌ 未找到包含 '{name}' 的宝可梦")
            print("💡 提示：请尝试使用英文名称，如 'Charizard', 'Pikachu'")
            return
        
        # 处理多个匹配结果
        if len(matches) > 1:
            print(f"⚠️ 找到 {len(matches)} 只匹配的宝可梦，显示前5个：")
            display(matches[['name', 'forme', 'stage']].head(5))
            print("\n将针对第一只进行详细预测...\n")
        
        # 取第一个匹配项进行预测
        row = matches.iloc[0]
        
        # 显示详细信息
        print(f"📋 宝可梦信息：")
        print(f"   名称: {row['name']}")
        print(f"   形态: {row['forme'] if row['forme'] else '标准形态'}")
        print(f"   HP: {row['hp']:.0f}")
        print(f"   最高攻击: {row['best_atk']:.0f}")
        print(f"   最高防御: {row['best_def']:.0f}")
        print(f"   速度: {row['spe']:.0f}")
        print(f"   实际进化阶段: {row['stage']}")
        print("\n" + "-" * 40)
        predict_and_display(row['hp'], row['best_atk'], row['best_def'], row['spe'])

btn_name.on_click(on_click_name)
display(widgets.VBox([name_input, btn_name, out_name]))

In [ ]:
# ========== 方式3：随机抽取预测 ==========
import random

print("\n" + "=" * 60)
print("🎲 预测方式3：随机抽取宝可梦")
print("=" * 60)

# 创建随机抽取控件
btn_random = widgets.Button(description='🎰 随机抽取一只', button_style='warning')
out_random = widgets.Output()

def on_click_random(b):
    """随机抽取按钮回调函数"""
    with out_random:
        clear_output(wait=True)
        
        # 随机采样1个样本
        row = df.sample(1).iloc[0]
        
        # 显示宝可梦信息
        print(f"🎲 随机选中的宝可梦：")
        print(f"   名称: {row['name']}")
        print(f"   形态: {row['forme'] if row['forme'] else '标准形态'}")
        print(f"   属性: {' / '.join(row['types']) if row['types'] else '未知'}")
        print(f"\n   📊 种族值：")
        print(f"      HP: {row['hp']:.0f} | 攻击: {row['atk']:.0f} | 防御: {row['def']:.0f}")
        print(f"      特攻: {row['spa']:.0f} | 特防: {row['spd']:.0f} | 速度: {row['spe']:.0f}")
        print(f"      总和(BST): {row['bst']:.0f}")
        print(f"\n   实际进化阶段: {row['stage']}")
        print("\n" + "-" * 40)
        predict_and_display(row['hp'], row['best_atk'], row['best_def'], row['spe'])

btn_random.on_click(on_click_random)
display(widgets.VBox([btn_random, out_random]))

In [ ]:
# ============================================================================
# 第十一部分：过拟合检测与泛化能力评估
# ============================================================================

# 重新计算以确保数据一致性
y_train_pred = model.predict(X_train)
y_test_pred = model.predict(X_test)

train_acc = accuracy_score(y_train, y_train_pred)
test_acc = accuracy_score(y_test, y_test_pred)
diff = train_acc - test_acc  # 泛化差距

print("🔬 模型泛化能力检测：")
print("=" * 50)
print(f"训练集准确率: {train_acc:.4f} ({train_acc*100:.2f}%)")
print(f"测试集准确率: {test_acc:.4f} ({test_acc*100:.2f}%)")
print(f"泛化差距: {diff:+.4f} ({abs(diff)*100:.2f}%)")
print()

# 判断是否存在过拟合
# 经验法则：如果训练集比测试集准确率高10%以上，可能存在过拟合
if diff > 0.1:
    print("⚠️  警告：检测到可能的过拟合现象！")
    print("   建议：")
    print("   1. 增加正则化强度（调整C参数）")
    print("   2. 收集更多训练数据")
    print("   3. 尝试特征选择减少维度")
    print("   4. 使用交叉验证评估稳定性")
elif diff < -0.05:
    print("ℹ️  提示：测试集表现略优于训练集")
    print("   这在数据量较小时可能出现，属于正常波动")
else:
    print("✅ 优秀：模型泛化能力良好！")
    print("   训练集与测试集表现接近，无明显过拟合或欠拟合迹象")

In [ ]:
# ============================================================================
# 第十二部分：批量预测评估系统
# ============================================================================

# 验证模型支持概率输出
if not hasattr(model, 'predict_proba'):
    raise AttributeError("错误：当前模型不支持概率预测。请确保使用LogisticRegression等支持predict_proba的分类器。")

# 创建批量评估界面
btn_sample = widgets.Button(description='🎲 随机抽取10只进行批量评估', button_style='info')
out_sample = widgets.Output()

def on_click_sample(b):
    """
    批量评估函数：随机抽取10只宝可梦进行预测评分
    
    评分规则：
    - Top-1预测正确：+10分
    - Top-2预测正确：+5分（体现模型的次优判断能力）
    - 都不正确：0分
    - 满分：100分（10只全部Top-1正确）
    """
    with out_sample:
        clear_output(wait=True)
        
        # 随机抽样（每次运行不同结果）
        sample = df.sample(10, random_state=None)
        classes = model.classes_
        total_score = 0
        rows = []
        
        print("📊 批量预测评估结果")
        print("=" * 80)
        
        for idx, (_, row) in enumerate(sample.iterrows(), 1):
            # 构造特征输入
            X_input = pd.DataFrame(
                [[row[col] for col in feature_cols]],
                columns=feature_cols
            )
            
            # 获取预测概率
            proba = model.predict_proba(X_input)[0]
            
            # 按概率排序，获取Top-2预测
            sorted_indices = np.argsort(proba)[::-1]  # 降序排列
            first_pred = classes[sorted_indices[0]]  # 最可能类别
            second_pred = classes[sorted_indices[1]]  # 第二可能类别
            
            # 与真实标签对比并计分
            actual = row['stage']
            score = 0
            if actual == first_pred:
                score = 10  # Top-1命中
                mark = "✅"
            elif actual == second_pred:
                score = 5   # Top-2命中
                mark = "⚠️"
            else:
                mark = "❌"
            
            total_score += score
            
            # 记录详细结果
            rows.append({
                '#': idx,
                '名称': row['name'],
                '形态': row['forme'] if row['forme'] else '-',
                'HP': int(row['hp']),
                '最高攻击': int(row['best_atk']),
                '最高防御': int(row['best_def']),
                '速度': int(row['spe']),
                '实际阶段': actual,
                '第一预测': f"{first_pred} {mark}" if mark == '✅' else first_pred,
                '第二预测': f"{second_pred} {mark}" if mark == '⚠️' else second_pred,
                '得分': f"{score}/10"
            })
        
        # 创建结果DataFrame并美化显示
        result_df = pd.DataFrame(rows)
        styled = result_df.style.set_caption(
            "🎯 随机10只宝可梦预测对比（含Top-2预测）"
        ).hide(axis='index')
        
        display(styled)
        
        # 输出总结统计
        print("\n" + "=" * 80)
        print(f"🏆 总得分：{total_score} / 100 分")
        print(f"📈 准确率：{total_score/10:.1f}% （仅计Top-1正确）")
        print("\n📖 评分规则说明：")
        print("   • Top-1预测完全正确：+10分 ✅")
        print("   • Top-2预测正确（第一预测错误但第二正确）：+5分 ⚠️")
        print("   • Top-2都错误：0分 ❌")
        
        # 性能评级
        if total_score >= 90:
            rating = "🌟🌟🌟 优秀"
        elif total_score >= 70:
            rating = "🌟🌟 良好"
        elif total_score >= 50:
            rating = "🌟 及格"
        else:
            rating = "需改进"
        print(f"\n性能评级：{rating}")

btn_sample.on_click(on_click_sample)
display(btn_sample, out_sample)

---

## 项目总结

### 🎯 实现成果

本成功实现了基于逻辑回归的宝可梦进化阶段预测系统，具备以下特点：

1. **完整的数据处理流程**：从原始CSV到特征工程的端到端处理
2. **创新的进化图算法**：使用BFS全局计算进化深度，处理复杂进化关系
3. **实用的特征工程**：通过best_atk/best_def简化特征空间，保留核心信息
4. **稳健的建模方法**：Pipeline+标准化+平衡权重，确保模型可靠性
5. **丰富的交互功能**：三种预测方式满足不同使用场景
6. **全面的评估体系**：准确率、混淆矩阵、批量测试等多维评估

### 🔬 技术亮点

- **图论应用**：将进化关系抽象为有向图，使用BFS计算拓扑层级
- **特征构造**：从6维原始特征提炼出4维高信息量特征
- **类别不平衡处理**：使用class_weight='balanced'自适应调整
- **概率输出**：不仅给出预测结果，还提供置信度和备选方案

### 📝 使用建议

1. **数据要求**：需要Pokemon_Showdown_pokedex.csv在同一目录
2. **环境配置**：Python 3.11+，安装requirements.txt中的依赖
3. **运行顺序**：按Notebook单元格顺序依次执行
4. **自定义预测**：可修改feature_cols调整使用的特征

### 🔮 未来改进方向

- [ ] 引入更多特征（如身高、体重、蛋组等）
- [ ] 尝试其他算法（Random Forest、XGBoost、神经网络）
- [ ] 添加超参数自动调优（GridSearchCV）
- [ ] 实现交叉验证获得更稳定的性能估计
- [ ] 开发Web应用接口供在线使用
- [ ] 添加更多可视化（决策边界、PCA降维展示等）

---

**项目作者**: KcirtW0009  
**联系方式**: KcirtW0009@outlook.com  
**GitHub**: [KcirtW0009/LogisticRegressionPrediction-EvolutionaryStage-by-PokemonStats](https://github.com/KcirtW0009/LogisticRegressionPrediction-EvolutionaryStage-by-PokemonStats)

---
*最后更新时间: 2026-05-26*